In [ ]:
import Pkg 
Pkg.activate("./..")

In [ ]:
import QuantumGrav as QG
import Flux 
import CUDA 
import GraphNeuralNetworks as GNN
import CausalSets as CS
import Arrow
import Tables

We use `GraphNeuralNetworks.jl` for building the variational autoencoder. This package has many kinds of graph-specific layers already implemented, see the [docs fo more](https://juliagraphs.org/GraphNeuralNetworks.jl/docs/GraphNeuralNetworks.jl/stable/api/basic/). In order to use it, we first need to find out: 
- which data to use 
- how to turn the tabular data into the datatype the GNN package needs
- how to use the GNN package with batches 

In [ ]:
path_to_data = joinpath("/", "home", "hmack", "Data", "QuantumGrav", "small_data_2D", )

In [ ]:
list_of_files = filter(x -> occursin(".arrow", x),readdir(path_to_data));
list_of_files[1:5]

### First idea for which data to use 
The primary information that we use is in the graph structure, not the nodes individually. However, in order to augment the system with more information, we can try and use the coordinates and the past and future relations on each node and treat them as node features. This way, we pass the network information about the causal sets: 
- the manifold they are embedded in 
- its dimensionality 
- the structure of the graph from node to node

We need to first find a good way to encode this information and pass it to the network. 

In [ ]:
first_batch = Arrow.Table(joinpath(path_to_data, list_of_files[1]));

features to use for now: coords, linkMatrix, past relations, future relations, coords

In [ ]:
past_relations = first_batch.past_relations;
future_relations = first_batch.future_relations;
coords = first_batch.coords;
linkMatrix = first_batch.linkMatrix;
atomCount = Int.(first_batch.n);
nmax = Int.(first_batch.nmax[1]);

In [ ]:
to_matrix(x) = hcat(x...); 

In [ ]:
function collate_matrix(x::Matrix{T}, n::Int64, m::Int64)::Matrix{T} where T 
    mat = zeros(T, n, m);

    n_, m_ = size(x);
    mat[1:n_, 1:m_] = x;
    return mat;
end

In [ ]:
function encode_data(n_max, linkMatrix, past_relations, future_relations, coords)::GNN.GNNGraph

    n_nodes = size(past_relations, 1)

    linkMatrix = collate_matrix(reshape(linkMatrix, (n_nodes, n_nodes)), n_max, n_max) 

    past_relations = collate_matrix(reshape(past_relations, n_nodes,:), n_max, n_max)

    future_relations = collate_matrix(reshape(future_relations, n_nodes,:), n_max, n_max)

    coords = collate_matrix(reshape(coords, :, n_nodes), 2, n_max)
    
    return GNN.GNNGraph(linkMatrix, ndata = (past_relations=past_relations, future_relations=future_relations, coords=coords))
end

pay attention to the vectorization syntax in Julia func.(v) vectorizes func over each elements of v. this is easily missed.

In [ ]:
encoded = encode_data.(nmax, to_matrix.(linkMatrix), to_matrix.(past_relations), to_matrix.(future_relations), to_matrix.(coords));

In [ ]:
first(encoded)

In [ ]:
GNN.batch(encoded)

### The simplest way: Graph convolutional network

We do this by hand for now, and later we can use `AutoEncoderToolkit.jl`. Graph-convolutional networks only use the adjacency-, i.e., link matrix of the graph, and hence are only structural in nature.

**Remark:** Message-passing networks (underlying all GNNs and actually also other stuff like CNNs) have the same discriminative power as the Weisfeiler-Lehman Graph Isomorphism Test. Hence some non-isomorphic graphs cannot be distinguished by them. Will this be a problem? 

We use a simpler approach for the data first - only use the adjacency matrix

In [ ]:
function encode_data(n_max, n_nodes, linkMatrix)::GNN.GNNGraph

    linkMatrix = collate_matrix(reshape(collect(linkMatrix), (n_nodes, n_nodes)), n_max, n_max)
    
    # we can construct other node features here if we need them, like we did before or in another way
    # here we just us ones because the damn package doesn't work with empty node features

    # more meaningful node features: 
    
    # 1. Causal layer (approximate using in/out degree)
    # in_degree = vec(sum(linkMatrix, dims=1))
    # out_degree = vec(sum(linkMatrix, dims=2))
    
    # # 2. Temporal position estimation
    # # In a DAG, this can approximate position in causal ordering
    # temporal = zeros(Float32, n_max)
    # for i in 1:n_nodes
    #     temporal[i] = out_degree[i] / (in_degree[i] + 1)  # Add 1 to avoid division by zero
    # end

    return GNN.GNNGraph(linkMatrix; ndata = vcat(ones(Float32, n_nodes), zeros(Float32, n_max - n_nodes)))
end

use a subset of the data to avoid running out of memory

In [ ]:
using StatsBase 

In [ ]:
dset = QG.DataLoader.Dataset(
    path_to_data, 
    sample(list_of_files, Int(ceil(length(list_of_files)/10))),
    cache_size = 5,
    columns = [:past_relations, :future_relations, :coords, :linkMatrix, :n, :nmax],
)

In [ ]:
g = encode_data(nmax, Int(dset[1].n), dset[1].linkMatrix)

In [ ]:
fieldnames(typeof(g))

In [ ]:
GNN.adjacency_matrix(g)

the Flux/MLUtils dataloader is almost useless here because it attempts to load all data at once. What's the point of this thing?? Just use it for now because we are operating on small datasets. Maybe use `DTables.jl` later?
Better version is to generate the data randomly from the start

In [ ]:
dset = QG.DataLoader.Dataset(
    path_to_data, 
    sample(list_of_files, Int(ceil(length(list_of_files)/10))),
    cache_size = 5,
    columns = [:past_relations, :future_relations, :coords, :linkMatrix, :n, :nmax],
)
shuffle_loader = Flux.DataLoader(
    dset,
    batchsize = 128,
    shuffle = true,
)

hence we operate only on the dataset, which does not do this.

graph batches then can be made like this: 

### build a simple VAE

In [ ]:
mutable struct VAE{E, D}
    encoder::E
    latent::Flux.Parallel
    decoder::D

end


function VAE(input_dim, latent_dim, output_dim; encoder_hidden_dims = [128, 80], decoder_hidden_dims = [80, 240, 680,], activation = relu)

    encoder = GNN.GNNChain(
        GNN.GCNConv(input_dim => encoder_hidden_dims[1] , activation),
        GNN.GCNConv(encoder_hidden_dims[1] => encoder_hidden_dims[2], activation),
        GNN.GlobalPool(mean),
    )

    latent = Flux.Parallel(
        tuple, 
        Flux.Dense(encoder_hidden_dims[2], latent_dim, identity), # Mean, no activation to not restrict expressivity of mean 
        Flux.Dense(encoder_hidden_dims[2], latent_dim, identity)  # Log variance, no activation to not restrict expressivity of log(sigma) 
    )

    decoder = Flux.Chain(
        Flux.Dense(latent_dim, decoder_hidden_dims[1], activation),
        Flux.Dense(decoder_hidden_dims[1], decoder_hidden_dims[2], activation),
        Flux.Dense(decoder_hidden_dims[2], decoder_hidden_dims[3], activation),
        Flux.Dense(decoder_hidden_dims[3], output_dim, Flux.sigmoid) # Output reconstruction
    )

    return VAE(encoder, latent, decoder)

end

Flux.@layer VAE

# reparameterization trick to be able to differentiate through the sampling process
function reparameterize(µ::Array{Float32, N}, logσ²::Array{Float32, N})::Array{Float32, N} where N
    # Sample from the standard normal distribution
    ε = randn(size(µ))
    
    # Reparameterization trick
    z = µ .+ exp.(0.5f0 .* logσ²) .* ε

    return z
end

# Forward pass through the VAE
function (vae::VAE)(g::GNN.GNNGraph, x::AbstractArray)
    # Encode input to get latent distribution parameters
    (µ, logσ²) = vae.encoder(g, x) |> vae.latent
    
    # Sample z from the latent distribution
    z = reparameterize(µ, logσ²)
    
    # Decode z to get reconstruction
    x̂ = vae.decoder(z)
    
    return x̂, µ, logσ²
end


# VAE loss function: reconstruction loss + KL divergence
function elbo_vae_loss(g::GNN.GNNGraph, x::AbstractArray, model::VAE; beta::Float32 = 1.0f0)
    # Get the true adjacency matrix from the graph
    adj_true = GNN.adjacency_matrix(g) # This is the ground truth adjacency matrix
    
    # Flatten it to match decoder output format
    adj_true_flat = vec(adj_true)
    
    # Forward pass through model - x is just node features, not used in loss
    adj_pred_flat, μ, logσ² = model(g, g.ndata.x)

    # Reconstruction loss
    rec_loss = Flux.logitbinarycrossentropy(adj_true_flat, adj_pred_flat, agg=sum) / size(x, 2)

    kl_div = 0.5f0 .* mean(sum(exp.(logs2) .+  µ .* µ .- 1f0 .- logs2, dims=1))
    # giving dim=2 here is not necessary anymore because we only have one dimension left

    return rec_loss + beta*kl_div, rec_loss, kl_div
end


In [ ]:
n_max = 30

In [ ]:
vae = VAE(30, 12, n_max*n_max; encoder_hidden_dims = [128, 80], decoder_hidden_dims = [80, 240, 680], activation = Flux.relu)

In [ ]:
x = rand(Float32, (30, 1));

In [ ]:
e = vae.encoder(g, x)


In [ ]:
l = vae.latent(e)


In [ ]:
z = reparameterize(l[1], l[2])


In [ ]:
x_ = vae.decoder(z)

In [ ]:
vae(g, x)

In [ ]:
# VAE loss function: reconstruction loss + KL divergence
function elbo_vae_loss(g::GNN.GNNGraph, x::AbstractArray, model::VAE; beta::Float32 = 1.0f0) where N
    # Get the true adjacency matrix from the graph
    adj_true = GNN.adjacency_matrix(g) # This is the ground truth adjacency matrix
    
    # # Flatten it to match decoder output format
    adj_true_flat = vec(adj_true)
    
    # # Forward pass through model - x is just node features, not used in loss
    adj_pred_flat, μ, logσ² = out = vae(g, reshape(g.ndata.x, (:, 1)))

    # # Reconstruction loss
    # rec_loss = Flux.logitbinarycrossentropy(adj_true_flat, adj_pred_flat, agg=sum) / size(x, 2)

    # kl_div = 0.5f0 .* mean(sum(exp.(logs2) .+  µ .* µ .- 1f0 .- logs2, dims=1))
    # # giving dim=2 here is not necessary anymore because we only have one dimension left

    # return rec_loss + beta*kl_div, rec_loss, kl_div
end

In [ ]:
GNN.adjacency_matrix(g)

In [ ]:
x

In [ ]:
reshape(g.ndata.x, (30, 1))

In [ ]:
out = vae(g, reshape(g.ndata.x, (30, 1)))

In [ ]:
elbo_vae_loss(g, reshape(g.ndata.x, (30, 1)), vae)

works in principle, now set up training loop

In [ ]:
# hyper parameters
input_dim = n_max 
latent_dim = 12
output_dim = n_max*n_max
encoder_hidden_dims = [128, 80]
decoder_hidden_dims = [80, 240, 680]
activation = Flux.relu # I am pretty sure this isn't right I need to use tanh or sigmoid here
batchsize = 64
epochs = 1000
beta = 1.0f0
learning_rate = 1e-2
# for early stopping
current_loss = Inf
last_loss = Inf
patience = 15 

# model
device = CUDA.functional() ? Flux.gpu : Flux.cpu 

vae = VAE(input_dim, latent_dim, output_dim; encoder_hidden_dims = encoder_hidden_dims, decoder_hidden_dims = decoder_hidden_dims, activation = activation) |> device

# dataset
dset = QG.DataLoader.Dataset(
    path_to_data, 
    sample(list_of_files, Int(ceil(length(list_of_files)/10))),
    cache_size = 5,
    columns = [:past_relations, :future_relations, :coords, :linkMatrix, :n, :nmax],
)

train_loader, valid_loader, test_loader = Flux.DataLoader.(Flux.splitobs(dset, at=(0.7, 0.15, 0.15))[1:3], 
    batchsize = batchsize,
    shuffle = true,
)

# optimizer
opt = Flux.setup(Flux.Adam(learning_rate), vae)

# training loop
for epoch in 1:epochs 
    println("Epoch: $epoch")
    for batch in train_loader 
        gbatch = GNN.batch([encode_data(n_max, Int(element.n), element.linkMatrix) for element in batch]) |> device

        # Compute gradient
        all_loss, grads = Flux.withgradient(vae) do vae
            elbo_vae_loss(gbatch, gbatch.ndata.x, vae; beta=beta)
        end

        # Check for numerical stability issues
        if any(isnan, all_loss)
            @warn "NaN detected in loss at epoch $epoch"
            hasnan = true
            break
        end
        
        # Update parameters
        Flux.update!(opt_state, vae, grads[1])
    end

    validation_loss = zeros(Float32, length(valid_loader))
    for (i, batch) in enumerate(valid_loader)
        gbatch = GNN.batch([encode_data(n_max, Int(element.n), element.linkMatrix) for element in batch]) |> gpu

        # Compute loss
        all_loss, rec_loss, kl_div = elbo_vae_loss(gbatch, gbatch.ndata.x, vae; beta=beta)
        validation_loss[i] = all_loss

        # Check for numerical stability issues
        if any(isnan, all_loss)
            @warn "NaN detected in loss at epoch $epoch"
            hasnan = true
            break
        end
    end

    current_loss = mean(validation_loss)

    if current_loss < last_loss*0.99
        patience = 15
        last_loss = current_loss
    else
        patience -= 1
        if patience == 0
            @info "Early stopping at epoch $epoch"
            break 
        end
    end
end